# 6.7 Implementing FM

The integrated FM formula, $\sin(2\pi f_c t + \tfrac{D}{f_m}\sin(2\pi f_m t))$, is easy to compute directly. But there is a more flexible and more general way to implement FM that connects the pieces we have built in this book:

1. In the previous sections we built a correct **time-varying oscillator** that accumulates phase, `osc(freq)`.
1. In [Chapter 3](../03-additive-synthesis) we built **wavetable synthesis** to make oscillators cheap.
1. FM is just a time-varying oscillator whose frequency signal happens to be _another oscillator_.

Combining these, we can write a general FM oscillator whose modulating signal can be **any sound at all**, not just a single sinusoid. This is far more expressive than the closed-form equation: the modulator can be a chord, a noise source, or even a recorded sample. The interactive example below lets you explore FM by editing the carrier frequency, modulating frequency, and index of modulation, then listening to and plotting the result. Try to reproduce a bright harmonic tone, then an inharmonic bell.

In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq

F_S = 44100


def fm_tone(f_c, f_m, I, dur=2.0):
    """Classic FM: carrier f_c, modulator f_m, index of modulation I."""
    t = np.arange(int(dur * F_S)) / F_S
    x = np.sin(2 * np.pi * f_c * t + I * np.sin(2 * np.pi * f_m * t))
    return pq.Audio(x.astype(np.float32), F_S)


def show_fm(f_c, f_m, I):
    audio = fm_tone(f_c, f_m, I)
    audio.normalize(peak_dbfs=-6.0)
    # Plot the amplitude spectrum so we can see the sidebands.
    x = np.asarray(audio.samples).reshape(-1)
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    freqs = np.fft.rfftfreq(len(x), 1 / F_S)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(freqs, X / X.max(), color="#007BC0")
    ax.axvline(f_c, color="#C41230", ls="--", lw=1, alpha=0.7)
    ax.set_xlim(0, f_c + (I + 3) * f_m + 100)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Amplitude")
    plt.show()
    return pq.play(audio)


In [ ]:
# Edit these three parameters, then run the cell to hear and see the result.
f_c = 440   # carrier frequency (Hz): sets the pitch
f_m = 110   # modulating frequency (Hz): the ratio f_c / f_m sets harmonicity
I = 2       # index of modulation: higher means brighter, with more sidebands

show_fm(f_c, f_m, I)
